In [17]:
import os
import numpy as np
import librosa
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

In [18]:
DATA_PATH = "Baby crying"
SR = 16000             # sampling rate
DURATION = 3           # seconds
SAMPLES = SR * DURATION
N_MELS = 64



# Load and preprocess one file

In [19]:
def preprocess_file(file_path):
    signal, _ = librosa.load(file_path, sr=SR)
    if len(signal) < SAMPLES:
        signal = np.pad(signal, (0, SAMPLES - len(signal)))
    else:
        signal = signal[:SAMPLES]

    mel = librosa.feature.melspectrogram(y=signal, sr=SR, n_mels=N_MELS)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = mel_db.T  # (time, n_mels)
    return mel_db

# Build dataset

In [20]:
def load_dataset(data_path):
    X, y, labels = [], [], []
    class_labels = sorted(os.listdir(data_path))
    label_map = {lab: i for i, lab in enumerate(class_labels)}

    for label in class_labels:
        folder = os.path.join(data_path, label)
        for fname in os.listdir(folder):
            if not fname.endswith(".wav"): 
                continue
            mel = preprocess_file(os.path.join(folder, fname))
            X.append(mel)
            y.append(label_map[label])
    return np.array(X, dtype="float32"), np.array(y), class_labels

X, y, class_labels = load_dataset(DATA_PATH)


# Pad sequences to same length

In [21]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
max_len = max([m.shape[0] for m in X])
X = pad_sequences(X, maxlen=max_len, padding="post", dtype="float32")
X = np.expand_dims(X, -1)  # add channel
y = to_categorical(y, num_classes=len(class_labels))

# Split

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)


# Model: CNN + LSTM

In [23]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(max_len, N_MELS, 1)),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Reshape((-1, 64)),          # reshape for LSTM
    layers.LSTM(128),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(class_labels), activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


/Volumes/CrucialX9/Project/venv/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 92, 62, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 46, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 44, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 22, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 308, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 9)              │         1,161 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 135,305 (528.54 KB)

 Trainable params: 135,305 (528.54 KB)

 Non-trainable params: 0 (0.00 B)

# Train

In [24]:
history = model.fit(X_train, y_train, validation_split=0.2, epochs=30, batch_size=32)


Epoch 1/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 7s 47ms/step - accuracy: 0.1350 - loss: 2.2189 - val_accuracy: 0.1180 - val_loss: 2.1447
Epoch 2/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 42ms/step - accuracy: 0.1478 - loss: 2.1772 - val_accuracy: 0.1370 - val_loss: 2.0901
Epoch 3/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 40ms/step - accuracy: 0.1875 - loss: 2.0979 - val_accuracy: 0.2250 - val_loss: 1.9644
Epoch 4/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 42ms/step - accuracy: 0.2258 - loss: 2.0258 - val_accuracy: 0.2280 - val_loss: 1.9233
Epoch 5/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 40ms/step - accuracy: 0.2773 - loss: 1.8663 - val_accuracy: 0.2960 - val_loss: 1.7635
Epoch 6/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 39ms/step - accuracy: 0.3231 - loss: 1.7443 - val_accuracy: 0.3810 - val_loss: 1.5960
Epoch 7/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 37ms/step - accuracy: 0.3436 - loss: 1.6868 - val_accuracy: 0.3710 - val_loss: 1.5659
Epoch 8/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 41ms/step - accuracy: 0.3643 - loss: 1.6244 - val_accu

# Evaluate

In [25]:
loss, acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {acc:.2f}")

40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.5152 - loss: 1.1936
Test Accuracy: 0.52
